<a href="https://colab.research.google.com/github/zamanmiraz/predict-crypto/blob/main/sentiment/Bitcoin_Sentiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
gauravduttakiit_bitcoin_tweets_16m_tweets_with_sentiment_tagged_path = kagglehub.dataset_download('gauravduttakiit/bitcoin-tweets-16m-tweets-with-sentiment-tagged')
# andreapenasmartinez_bitcoin_twitter_sentiment_dataset_20132023_path = kagglehub.dataset_download('andreapenasmartinez/bitcoin-twitter-sentiment-dataset-20132023')

print('Data source import complete.')


In [ ]:
import torch
if torch.cuda.is_available():
    device = torch.device('cuda')
else:
    device = torch.device('cpu')

print(device)

In [ ]:
! pip install gensim
! pip install torch==2.3.0 torchtext==0.18.0

In [ ]:
import pandas as pd
from gensim.models import Word2Vec
from torchtext.vocab import build_vocab_from_iterator
import torch
import torch.nn as nn
import numpy as np

In [ ]:
df = pd.read_csv(
    f"{gauravduttakiit_bitcoin_tweets_16m_tweets_with_sentiment_tagged_path}/mbsa.csv",
    nrows=500000,        # try 200k first, increase gradually
    on_bad_lines='skip',
    engine='python'
)

In [ ]:
df.head()

In [ ]:
# Drop missing or NaN text
df = df.dropna(subset=['text'])
# Tokenize each tweet into a list of words
sentences = [str(text).lower().split() for text in df['text']]

In [ ]:
# Dividing df into train and test
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

In [ ]:
from gensim.models import Word2Vec

embedding_dim = 150

word2vec_model = Word2Vec(
    sentences,
    vector_size=embedding_dim,
    window=5,
    min_count=2,   # ignore very rare words
    workers=4
)

In [ ]:
max_vocab_size = 25000

def yield_tokens(sentences):
    for sent in sentences:
        yield sent

vocab = build_vocab_from_iterator(
    yield_tokens(sentences),
    max_tokens=max_vocab_size,
    specials=['<unk>', '<pad>']
)

unk_idx = vocab['<unk>']
pad_idx = vocab['<pad>']
vocab.set_default_index(unk_idx)


In [ ]:
# -------------------------------
# 3. Create embedding matrix
# -------------------------------
def create_embedding_matrix(vocab, word2vec_model, embedding_dim):
    embedding_matrix = np.zeros((len(vocab), embedding_dim))
    for word, idx in vocab.get_stoi().items():
        if word in word2vec_model.wv:
            embedding_matrix[idx] = word2vec_model.wv[word]
        else:
            embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))
    return embedding_matrix

embedding_matrix = create_embedding_matrix(vocab, word2vec_model, embedding_dim)

# Convert to torch tensor for nn.Embedding
embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)

In [ ]:
# -------------------------------
# 4. Define Embedding Layer
# -------------------------------
embedding_layer = nn.Embedding.from_pretrained(
    embeddings=embedding_matrix,
    freeze=False,          # True = keep pretrained fixed, False = allow fine-tuning
    padding_idx=pad_idx
)

print("Vocab size:", len(vocab))
print("Embedding layer shape:", embedding_layer.weight.shape)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm.auto import tqdm
from torchtext.data.functional import to_map_style_dataset
from torch.utils.data import DataLoader
import numpy as np
import pandas as pd # Import pandas to explicitly show Series type handling

max_seq_len = 50

# Convert the dataset to map-style
train_dataset = [row for _, row in train_df.iterrows()]
test_dataset = [row for _, row in test_df.iterrows()]

# Tokenization function
def tokenize_text(text):
    # Ensure text is treated as a string before splitting
    return str(text).lower().split()

# Numericalization and padding function
def collate_batch(batch):
    label_list, text_list = [], []
    # Expecting batch to be a list of pandas Series (rows from the DataFrame)
    for sample in batch:
        # Access text and label by column names from the pandas Series
        # Add a check to ensure 'text' and 'Sentiment' columns exist and sample is a Series
        if isinstance(sample, pd.Series) and 'text' in sample and 'Sentiment' in sample:
            _text = sample['text']
            _label = sample['Sentiment']
        else:
            # If the sample is not in the expected format, skip or handle appropriately
            print(f"Skipping sample due to unexpected format: {type(sample)}, content: {sample}")
            continue # Skip this sample

        # Convert label: Map 'Positive' to 1 and 'Negative' to 0
        if _label == 'Positive':
            label_list.append(1)
        elif _label == 'Negative':
            label_list.append(0)
        else:
            # Handle other possible sentiment labels if necessary, or skip this sample
            continue # Skipping samples with unexpected sentiment labels

        processed_text = torch.tensor(
            [vocab[token] for token in tokenize_text(_text)[:max_seq_len]],
            dtype=torch.int64
        )
        text_list.append(processed_text)

    # Handle the case where text_list is empty after filtering
    if not text_list:
        # Return empty tensors of appropriate shape and type
        # Ensure the second tensor has the sequence length dimension even if 0 batch size
        return torch.tensor([], dtype=torch.int64), torch.empty(0, max_seq_len, dtype=torch.int64)


    # Pad sequences
    padded_text_list = torch.nn.utils.rnn.pad_sequence(
        text_list, batch_first=True, padding_value=pad_idx
    )

    return torch.tensor(label_list, dtype=torch.int64), padded_text_list


# Create DataLoaders
batch_size = 64

train_dataloader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=collate_batch
)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False,
    collate_fn=collate_batch
)

print("Train DataLoader created.")
print("Test DataLoader created.")

# -------------------------------
# 4. Define EncoderGRU
# -------------------------------

latent_dim = 512
output_dim = 2


class EncoderGRU(nn.Module):
    def __init__(self, embedding_layer, latent_dim, dropout_rate=0.3):
        super().__init__()
        self.embedding = embedding_layer # embedding matrix into embedding layer
        self.dropout = nn.Dropout(dropout_rate) # Added Dropout
        self.rnn = nn.GRU(
            embedding_layer.embedding_dim,
            latent_dim,
            batch_first=True,
        )

    def forward(self, text):
        # text shape: (batch_size, seq_len)
        embedded = self.embedding(text)
        embedded = self.dropout(embedded) # Apply dropout after embedding
        # embedded shape: (batch_size, seq_len, embedding_dim)
        output, hidden = self.rnn(embedded)
        # output = self.dropout(output) # Apply dropout after RNN
        # output shape: (batch_size, seq_len, latent_dim)
        # hidden shape: (1, batch_size, latent_dim)
        # hidden.squeeze: (Batch Size and Latent Dimension)
        return output, hidden.squeeze(0)

encoder = EncoderGRU(embedding_layer, latent_dim)
print("Encoder model created with GRU and Dropout.")

# -------------------------------
# 5. Define CrossAttention
# -------------------------------

class CrossAttention(nn.Module):
    def __init__(self, latent_dim, dropout_rate=0.3):
        super().__init__()
        self.mha = nn.MultiheadAttention(embed_dim=latent_dim, num_heads=1, batch_first=True)
        self.normalization = nn.LayerNorm(latent_dim)
        self.dropout = nn.Dropout(dropout_rate) # Added Dropout

    def forward(self, x, context):
        attn_output, attn_score = self.mha(x, context, context)
        # Dimension:
        x = x + attn_output
        x = self.normalization(x)
        x = self.dropout(x) # Apply dropout after normalization
        return x

cross_attention = CrossAttention(latent_dim)
print("CrossAttention model created with Dropout.")

# -------------------------------
# 6. Define Classifier (Decoder)
# -------------------------------

class Classifier(nn.Module):
    def __init__(self, latent_dim, output_dim, dropout_rate=0.5):
        super().__init__()
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(latent_dim, output_dim)

    def forward(self, hidden_state):
        hidden_state = self.dropout(hidden_state)
        return self.fc(hidden_state)   # raw logits

classifier = Classifier(latent_dim, output_dim)
print("Classifier model created with Dropout.")

# -------------------------------
# 7. Combine Encoder and Classifier
# -------------------------------
class SentimentClassifier(nn.Module):
    def __init__(self, encoder, cross_attention, classifier): # Added cross_attention as parameter
        super().__init__()
        self.encoder = encoder
        self.cross_attention = cross_attention # Use the passed in cross_attention module
        self.classifier = classifier

    def forward(self, text):
        encoder_outputs, encoder_state = self.encoder(text)
        query = encoder_state.unsqueeze(1)    # (batch, 1, latent_dim)
        context = encoder_outputs             # (batch, seq_len, latent_dim)
        attn_output = self.cross_attention(query, context) # Pass through the cross_attention module
        attn_output = attn_output[:, 0, :]   # (batch, latent_dim)
        prediction = self.classifier(attn_output)
        return prediction

model = SentimentClassifier(encoder, cross_attention, classifier) # Pass cross_attention instance
model.to(device) # Move model to the appropriate device (CPU or GPU)
print("SentimentClassifier model created and moved to device.")

# -------------------------------
# 8. Define Loss Function and Optimizer
# -------------------------------
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), weight_decay=1e-4)
print("Loss function and Optimizer defined.")

# -------------------------------
# 9. Training Loop
# -------------------------------
def train(model, dataloader, optimizer, criterion, device):
    model.train()
    epoch_loss = 0
    processed_batches = 0
    for labels, text in tqdm(dataloader, desc="Training"):
        if text.size(0) == 0: # Skip empty batches
            continue

        labels = labels.to(device)
        text = text.to(device)

        optimizer.zero_grad()
        predictions = model(text)
        loss = criterion(predictions, labels)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        processed_batches += 1
    return epoch_loss / processed_batches if processed_batches > 0 else float('nan') # Avoid division by zero

# -------------------------------
# 10. Evaluation Loop
# -------------------------------
def evaluate(model, dataloader, criterion, device):
    model.eval()
    epoch_loss = 0
    correct_predictions = 0
    total_samples = 0
    processed_batches = 0
    with torch.no_grad():
        for labels, text in tqdm(dataloader, desc="Evaluating"):
            if text.size(0) == 0: # Skip empty batches
                continue

            labels = labels.to(device)
            text = text.to(device)

            predictions = model(text)
            loss = criterion(predictions, labels)
            epoch_loss += loss.item()

            # Calculate accuracy
            _, predicted = torch.max(predictions, 1)
            correct_predictions += (predicted == labels).sum().item()
            total_samples += labels.size(0)
            processed_batches += 1

    return epoch_loss / processed_batches if processed_batches > 0 else float('nan'), correct_predictions / total_samples if total_samples > 0 else 0.0


# -------------------------------
# 11. Train the model
# -------------------------------
N_EPOCHS = 30 # You can adjust the number of epochs
best_valid_loss = float('inf')

for epoch in range(N_EPOCHS):
    train_loss = train(model, train_dataloader, optimizer, criterion, device)
    valid_loss, valid_acc = evaluate(model, test_dataloader, criterion, device)

    if not np.isnan(valid_loss) and valid_loss < best_valid_loss: # Check for NaN before comparing
        best_valid_loss = valid_loss
        torch.save(model.state_dict(), 'best-model.pt')
        print(f"Epoch {epoch+1}: Validation loss improved. Saving model.")

    print(f"Epoch {epoch+1}: Train Loss: {train_loss:.3f}, Val Loss: {valid_loss:.3f}, Val Acc: {valid_acc:.3f}")

print("\nTraining finished.")